# XEduHub 模型库测试
## SOTA 预训练模型调用

本 Notebook 演示如何使用 XEduHub 模块调用 State-of-the-Art 预训练模型。

### 学习目标
- ✅ 了解常见 SOTA 模型架构
- ✅ 掌握 XEduHub Model Zoo 使用方法
- ✅ 快速调用预训练模型进行推理
- ✅ 理解迁移学习的基本概念

### 预计用时：15-20 分钟

In [ ]:
# 导入必要的库
import xeduhub
from xeduhub import ModelZoo
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
import numpy as np

print(f"XEduHub 版本：{xeduhub.__version__}")
print("✅ 库导入成功")

## 步骤 1: 浏览可用模型

查看 XEduHub 提供的预训练模型列表

In [ ]:
# 获取所有可用的模型
available_models = ModelZoo.list_models()

print("可用的预训练模型:")
print("=" * 60)
for idx, model_info in enumerate(available_models[:10], 1):
    print(f"{idx}. {model_info['name']}")
    print(f"   类型：{model_info['task']}")
    print(f"   描述：{model_info['description']}")
    print(f"   输入尺寸：{model_info['input_size']}")
    print()

print(f"共 {len(available_models)} 个模型可用")

## 步骤 2: 加载图像分类模型

In [ ]:
# 加载 ResNet-50 预训练模型
print("正在加载 ResNet-50 模型...")
classifier = ModelZoo.get_model(
    task_type='image_classification',
    model_name='resnet50'
)
print("✅ 模型加载完成")
print(f"   模型名称：ResNet-50")
print(f"   训练数据集：ImageNet-1k")
print(f"   Top-1 准确率：76.0%")
print(f"   Top-5 准确率：92.8%")

## 步骤 3: 准备测试图像

In [ ]:
# 从网络加载测试图像
image_urls = [
    'https://images.unsplash.com/photo-1543852786-1cf6624b9987?w=400',  # 猫
    'https://images.unsplash.com/photo-1560807707-8cc77767d783?w=400',  # 狗
    'https://images.unsplash.com/photo-1555617778-02518510b9fa?w=400'   # 汽车
]

def load_image_from_url(url):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    return img

# 加载第一张图像
test_image = load_image_from_url(image_urls[0])

# 显示图像
plt.figure(figsize=(8, 8))
plt.imshow(test_image)
plt.axis('off')
plt.title('Test Image')
plt.tight_layout()
plt.show()

print("✅ 测试图像加载完成")

## 步骤 4: 模型推理

In [ ]:
# 将 PIL 图像转换为 numpy 数组
image_array = np.array(test_image)

# 进行预测
print("正在进行图像识别...")
predictions = classifier.predict(image_array)

print("\n预测结果:")
print("=" * 50)
print(f"Top-1: {predictions['top1']['label']} ({predictions['top1']['confidence']:.2%})")
print(f"\nTop-5 预测:")
for i, pred in enumerate(predictions['top5'], 1):
    print(f"{i}. {pred['label']}: {pred['confidence']:.2%}")

# 可视化预测结果
plt.figure(figsize=(10, 6))
labels = [p['label'] for p in predictions['top5']]
confidences = [p['confidence'] for p in predictions['top5']]

bars = plt.bar(labels, confidences, color='#3498db')
plt.xlabel('Class Label')
plt.ylabel('Confidence')
plt.title('Top-5 Predictions')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)

# 在柱子上标注数值
for bar, conf in zip(bars, confidences):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{conf:.2%}', ha='center', va='bottom', rotation=90)

plt.tight_layout()
plt.savefig('prediction_result.png', dpi=300)
plt.show()

## 步骤 5: 测试不同模型

In [ ]:
# 测试不同的图像分类模型
models_to_test = [
    ('ResNet-18', 'resnet18'),
    ('ResNet-50', 'resnet50'),
    ('EfficientNet-B0', 'efficientnet_b0'),
    ('MobileNet-V2', 'mobilenet_v2')
]

results = []

for model_name, model_id in models_to_test:
    print(f"\n测试 {model_name}...")
    try:
        model = ModelZoo.get_model('image_classification', model_id)
        pred = model.predict(image_array)
        results.append({
            'Model': model_name,
            'Top1 Label': pred['top1']['label'],
            'Top1 Confidence': pred['top1']['confidence']
        })
        print(f"  ✅ Top-1: {pred['top1']['label']} ({pred['top1']['confidence']:.2%})")
    except Exception as e:
        print(f"  ❌ 加载失败：{e}")

# 对比结果
results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("不同模型预测结果对比:")
print(results_df)

## 步骤 6: 目标检测任务

In [ ]:
# 加载 YOLOv5 目标检测模型
print("加载 YOLOv5 目标检测模型...")
detector = ModelZoo.get_model(
    task_type='object_detection',
    model_name='yolov5s'
)
print("✅ 模型加载完成")

# 使用另一张包含多个物体的图像
street_image = load_image_from_url(
    'https://images.unsplash.com/photo-1449965408869-eaa3f722e40d?w=600'
)

plt.figure(figsize=(12, 8))
plt.imshow(street_image)
plt.axis('off')
plt.title('Street Scene for Object Detection')
plt.tight_layout()
plt.show()

# 进行目标检测
print("\n执行目标检测...")
detections = detector.predict(np.array(street_image))

print(f"检测到 {len(detections['boxes'])} 个物体:")
for i, (box, label, conf) in enumerate(zip(
    detections['boxes'], 
    detections['labels'], 
    detections['confidences']
), 1):
    print(f"{i}. {label} (置信度：{conf:.2%}) - 位置：{box}")

## 总结与拓展

### 本节所学
1. ✅ XEduHub Model Zoo 的基本用法
2. ✅ 加载和使用预训练模型
3. ✅ 图像分类和目标检测任务
4. ✅ 多模型对比评估

### 课后练习
1. 📝 尝试其他类型的模型（如语义分割、姿态估计）
2. 📝 使用自己的图像进行测试
3. 📝 了解模型量化和加速技术
4. 📝 探索迁移学习微调

### 积分奖励
- ✅ 完成基础练习：+150 XP
- ✅ 测试 3 个以上模型：+200 XP
- ✅ 完成目标检测：+150 XP

### 下一步
继续学习下一个 Notebook: `06_easytrain_no_code.ipynb`